In [34]:
import torch
import importlib
import sys
sys.path.append("../")

sys.path.append("../../")
import cumulant_analyzer
from cumulant_analyzer import CumulantAnalyzer, to_numpy, calculate_cumulants

from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
import numpy as np
import matplotlib.pyplot as plt
import plot_utils
from plot_utils import plot_comparison, plot_stats
from IPython.display import display, Markdown

import pandas as pd
pd.set_option('display.precision', 2)

import warnings
warnings.filterwarnings(
    "ignore",
    message="You are using `torch.load` with `weights_only=False`.*",
    category=FutureWarning,
)

torch.set_grad_enabled(False)
print("Disabled automatic differentiation")

from transformers.utils.logging import disable_progress_bar
disable_progress_bar()


%matplotlib inline

import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "notebook"  # or try "iframe" or "svg" or "png"

from IPython.display import display, HTML
import contextualization_utils
importlib.reload(contextualization_utils)
from contextualization_utils import display_tokens, create_df, create_interactive_token_display

Disabled automatic differentiation


## Analysis

In [35]:
ds = load_dataset("NeelNanda/pile-10k")['train']
filtered_indices = np.load('../filtered_indices.npy')
max_length = 128
model_name = 'gpt2-large'
analyzer = CumulantAnalyzer(model_name=model_name, max_length = 256, cache_mode = "huggingface")
tokenizer = analyzer.tokenizer

In [36]:
test_indx = np.random.choice(2244)
prompt_indx = filtered_indices[test_indx]
test_sequence = ds['text'][prompt_indx].strip()
data = create_df(analyzer, test_sequence, max_length = max_length)
all_probs, stats = data['all_probs'], data['stats']
data['df']

,avg_entropy,entropy_com,kld_center,κ₂,κ₃,κ₄,κ₅,κ₆
Free,3.30,5.75,2.45,0.72,0.56,0.51,0.45,0.35
Context,2.79,5.42,2.64,0.63,0.51,0.50,0.49,0.46
Free - Context,0.51,0.33,-0.19,0.09,0.06,0.01,-0.04,-0.11


<table style="font-size:20px; border-collapse: collapse;  margin-left: 0;">
  <tr>
    <th style="text-align:left; padding: 8px;">Metric</th>
    <th style="text-align:left; padding: 8px;">Comparison</th>
  </tr>
  <tr>
    <td style="padding: 8px;">Entropy</td>
    <td style="padding: 8px;">Free &gt; Context</td>
  </tr>
  <tr>
    <td style="padding: 8px;">Entropy of center</td>
    <td style="padding: 8px;">Free &gt; Context</td>
  </tr>
  <tr>
    <td style="padding: 8px;">KLD center</td>
    <td style="padding: 8px;">Context &gt; Free</td>
  </tr>
  <tr>
    <td style="padding: 8px;">&#954;<sub>2</sub></td>
    <td style="padding: 8px;">Free &gt; Context</td>
  </tr>
</table>

In [71]:
tokenizer.decode([250]).strip()

'�'

In [73]:

pd.set_option('display.precision', 3)
diff_np = np.abs(to_numpy((all_probs[0] - all_probs[1]).sum(0)))
k = 10
topk_indices = diff_np.argsort()[-k:][::-1]  # descending order

# Get token strings (assumes tokenizer is already loaded)
tokens = [repr(tokenizer.decode([i])) for i in topk_indices]
values = diff_np[topk_indices]

# Create DataFrame
df = pd.DataFrame({'token_id': topk_indices, 'token': tokens, 'Free COM': np_probs[0].sum(0)[topk_indices], 'Context COM': np_probs[1].sum(0)[topk_indices], 'abs_diff': values})
display(df)

,token_id,token,Free COM,Context COM,abs_diff
0,250,'�',0.660,2.784,2.123
1,251,'�',0.816,0.013,0.803
2,17118,' devil',0.002,0.555,0.552
3,447,'�',2.494,1.995,0.500
4,534,' your',1.237,1.687,0.451
5,13,'.',3.992,4.443,0.450
6,284,' to',2.318,1.877,0.441
7,1893,' president',0.138,0.568,0.430
8,531,' said',0.033,0.458,0.425
9,11,"','",3.554,3.129,0.424


In [38]:
# Function to get top-k predictions for each position
def get_topk_predictions(probs_array, tokenizer, k=3):
    """
    Get top-k token predictions for each position.
    
    Args:
        probs_array: Array of probabilities [seq_len, vocab_size]
        tokenizer: The tokenizer to decode token ids
        k: Number of top predictions to return
    
    Returns:
        List of strings with top-k predictions for each position
    """
    topk_info = []
    
    for i in range(probs_array.shape[0]):
        # Get top-k indices and their probabilities
        topk_probs, topk_indices = torch.topk(torch.tensor(probs_array[i]), k)
        
        # Format top-k predictions
        predictions = []
        for j in range(k):
            token = tokenizer.decode([topk_indices[j].item()])
            prob = topk_probs[j].item()
            predictions.append(f"{token} ({prob:.2%})")
        
        topk_info.append(" | ".join(predictions))
    
    return topk_info

def calculate_kl_divergence(p, q, epsilon=1e-12):
    """
    Calculate KL divergence D(P||Q) = sum(P * log(P/Q))
    
    Args:
        p (torch.Tensor): First probability distribution (P)
        q (torch.Tensor): Second probability distribution (Q) 
        epsilon (float): Small value to avoid log(0)
    
    Returns:
        torch.Tensor: KL divergence for each position/token
    """
    # Add small epsilon to avoid log(0)
    p_safe = p + epsilon
    q_safe = q + epsilon
    
    # Calculate KL divergence: D(P||Q) = sum(P * log(P/Q))
    kl_div = p_safe * np.log(p_safe / q_safe)
    
    # Sum over vocabulary dimension (last dimension)
    kl_div = kl_div.sum(axis=-1)
    
    return kl_div

In [43]:
# Convert probabilities to numpy
np_probs = np.array([to_numpy(all_probs[0]), to_numpy(all_probs[1])])

# Get tokens for the sequence
input_ids = tokenizer.encode(test_sequence.strip(), add_special_tokens=False, return_tensors="pt")
tokens = [tokenizer.decode([tid]) for tid in input_ids[0]]

# Get top-k predictions for both probability arrays
topk_with_context = get_topk_predictions(np_probs[0], tokenizer, k=3)
topk_no_context = get_topk_predictions(np_probs[1], tokenizer, k=3)

# Calculate KL divergence
kl_divergence = calculate_kl_divergence(np_probs[0], np_probs[1], epsilon=1e-12)

# Create the plot with enhanced hover information
fig = go.Figure()

# Prepare hover text with current token and top-k predictions for both conditions
hover_text = []
for i in range(len(tokens[max_length//2:max_length])):
    current_token = tokens[max_length//2 + i]
    hover_text.append(
        f"Token: {current_token}<br>"
        f"With context: {topk_with_context[i]}<br>"
        f"W/O context: {topk_no_context[i]}"
    )

fig.add_trace(go.Scatter(
    y=kl_divergence,
    name='KL Divergence',
    text=hover_text,
    hovertemplate='%{text}<br>KL Div: %{y:.3f}<extra></extra>',
    mode='lines+markers'
))

# Customize layout
fig.update_layout(
    title="KL Divergence with Top-3 Token Predictions (Context vs No Context)",
    xaxis_title="Position",
    yaxis_title="KL Divergence",
    hovermode='x unified',
    width=1000,
    height=600
)
display(HTML("<h4><b>Context</b></h4>"))
print(display_tokens(test_sequence, tokenizer, start_idx=0, end_idx=max_length//2))
create_interactive_token_display(test_sequence, tokenizer, all_probs, max_length, k=3)

# Show the figure
fig.show()

Donald Trump may be dancing with the devil in 2016 according to POLITICO's Roger Simon. | AP Photo Simon Says The Devil and Donald Trump

Alone in his bedroom on a dark and stormy night, Donald Trump was inventing some tax returns, when the devil appeared before him.

“Fear not


In [52]:
# Calculate KL divergence
entropy_np = to_numpy(stats['entropy'])

kld_center = to_numpy(stats['kld_center'])

# Create the plot with enhanced hover information
fig = go.Figure()

# Prepare hover text with current token and top-k predictions for both conditions
hover_text = []
for i in range(len(tokens[max_length//2:max_length])):
    current_token = tokens[max_length//2 + i]
    hover_text.append(
        f"Token: {current_token}<br>"
        f"With context: {topk_with_context[i]}<br>"
        f"W/O context: {topk_no_context[i]}"
    )

fig.add_trace(go.Scatter(
    y=entropy_np[0] - entropy_np[1],
    name='Free - Context',
    text=hover_text,
    hovertemplate='%{text}<br>Entropy: %{y:.3f}<extra></extra>',
    mode='lines+markers'
))

fig.add_trace(go.Scatter(
    y= - (kld_center[0] - kld_center[1]),
    name='Context - Free',
    text=hover_text,
    hovertemplate='%KLD Center: %{y:.3f}<extra></extra>',
    mode='lines+markers'
))

# Customize layout
fig.update_layout(
    title="Entropy",
    xaxis_title="Position",
    yaxis_title="KL Divergence",
    hovermode='x unified',
    width=1000,
    height=600
)
display(HTML("<h4><b>Context</b></h4>"))
print(display_tokens(test_sequence, tokenizer, start_idx=0, end_idx=max_length//2))
create_interactive_token_display(test_sequence, tokenizer, all_probs, max_length, k=3)

# Show the figure
fig.show()

Donald Trump may be dancing with the devil in 2016 according to POLITICO's Roger Simon. | AP Photo Simon Says The Devil and Donald Trump

Alone in his bedroom on a dark and stormy night, Donald Trump was inventing some tax returns, when the devil appeared before him.

“Fear not


In [64]:
# Calculate KL divergence
cumulants_np = to_numpy(stats['normalized_cumulants'])[:, 0]

# Create the plot with enhanced hover information
fig = go.Figure()

hover_text = []
for i in range(len(tokens[max_length//2:max_length])):
    current_token = tokens[max_length//2 + i]
    hover_text.append(
        f"Token: {current_token}<br>"
        # f"With context: {topk_with_context[i]}<br>"
        f"W/O context: {topk_no_context[i]}"
    )
fig.add_trace(go.Scatter(
    y=cumulants_np[1],
    name='No context',
    text=hover_text,
    hovertemplate='%{text}<br>Entropy: %{y:.3f}<extra></extra>',
    mode='lines+markers'
))

# Prepare hover text with current token and top-k predictions for both conditions
hover_text = []
for i in range(len(tokens[max_length//2:max_length])):
    current_token = tokens[max_length//2 + i]
    hover_text.append(
        # f"Token: {current_token}<br>"
        f"With context: {topk_with_context[i]}"
        #f"W/O context: {topk_no_context[i]}"
    )

fig.add_trace(go.Scatter(
    y=cumulants_np[0],
    name='Context',
    text=hover_text,
    hovertemplate='%{text}<br>Entropy: %{y:.3f}<extra></extra>',
    mode='lines+markers'
))



# Customize layout
fig.update_layout(
    title="KL Divergence with Top-3 Token Predictions (Context vs No Context)",
    xaxis_title="Position",
    yaxis_title="KL Divergence",
    hovermode='x unified',
    width=1000,
    height=600
)
display(HTML("<h4><b>Context</b></h3>"))
print(quick_display_tokens(test_sequence, tokenizer, start_idx=0, end_idx=max_length//2))

# display(HTML("<h4><b>Query</b></h3>"))
# print(quick_display_tokens(test_sequence, tokenizer, start_idx=max_length//2, end_idx=max_length))
create_interactive_token_display(test_sequence, tokenizer, all_probs, max_length, k=3)
# Show the figure
fig.show()

Q:

Class tufte-book and R listings code box width problem

When I run the following MWE:
\documentclass{tufte-book}

\usepackage{Sweavel}

\usepackage{amsthm}
\usepackage{math


### Analyzing the center

In [16]:

    
def cummean(arr):
    arr = to_numpy(arr)
    return np.cumsum(arr,  axis = 0) / np.expand_dims(np.arange(1, len(arr) + 1), 1)

cum_probs = np.array([cummean(all_probs[0]), cummean(all_probs[1])])

fig = go.Figure()
input_ids = tokenizer.encode(test_sequence.strip(), add_special_tokens=False, return_tensors="pt")
tokens = [tokenizer.decode([tid]) for tid in input_ids[0]]

fig.add_trace(go.Scatter(
    y=calculate_kl_divergence(cum_probs[0], cum_probs[1], epsilon=1e-12), 
    name='Entropy Diff',
    text=tokens[max_length//2:max_length],
    hovertemplate='Index: %{x}<br>Token: %{text}<br>KL Div: %{y:.3f}<extra></extra>'
))


### Analyzing the cumulants

In [17]:
fig = go.Figure()
input_ids = tokenizer.encode(test_sequence.strip(), add_special_tokens=False, return_tensors="pt")
tokens = [tokenizer.decode([tid]) for tid in input_ids[0]]

arr = to_numpy(stats['entropy'])
fig.add_trace(go.Scatter(
    y=arr[1] - arr[0], 
    name='Entropy Diff',
    text=tokens[max_length//2:max_length],
    hovertemplate='Index: %{x}<br>Token: %{text}<br>KL Div: %{y:.3f}<extra></extra>'
))

arr = to_numpy(stats['kld_center'])
fig.add_trace(go.Scatter(
    y=arr[0] - arr[1], 
    name='KL Diff',
    text=tokens[max_length//2:max_length],
    hovertemplate='Index: %{x}<br>Token: %{text}<br>KL Div: %{y:.3f}<extra></extra>'
))

arr = to_numpy(stats['normalized_cumulants'])
fig.add_trace(go.Scatter(
    y=arr[0, 0 , :] - arr[1, 0, :], 
    name='$\kappa_2$',
    text=tokens[max_length//2:max_length],
    hovertemplate='Index: %{x}<br>Token: %{text}<br>KL Div: %{y:.3f}<extra></extra>'
))

fig.add_trace(go.Scatter(
    y=arr[0, 1 , :] - arr[1, 1, :], 
    name='$\kappa_3$',
    text=tokens[max_length//2:max_length],
    hovertemplate='Index: %{x}<br>Token: %{text}<br>KL Div: %{y:.3f}<extra></extra>'
))

fig.update_layout(
    title='KL Divergence with Center: Context - No Context ',
    xaxis_title='Token Index',
    yaxis_title='KL Divergence'
)

fig.show()

In [29]:
# quick_fig = quick_kl_plot(arr[1] - arr[0], arr[1] - arr[0], test_sequence, analyzer.tokenizer)
arr = to_numpy(stats['kld_center'])
fig = go.Figure()
input_ids = tokenizer.encode(test_sequence.strip(), add_special_tokens=False, return_tensors="pt")
tokens = [tokenizer.decode([tid]) for tid in input_ids[0]]
fig.add_trace(go.Scatter(
    y=arr[0], 
    name='Context',
    text=tokens[max_length//2:max_length],
    hovertemplate='Token: %{text}<br>KL Div: %{y:.6f}<extra></extra>'
))

fig.add_trace(go.Scatter(
    y=arr[1], 
    name='No Context',
    text=tokens[max_length//2:max_length],
    hovertemplate='Token: %{text}<br>KL Div: %{y:.6f}<extra></extra>'
))
    
fig.show()

In [13]:
def calculate_kl_divergence(p, q, epsilon=1e-12):
    """
    Calculate KL divergence D(P||Q) = sum(P * log(P/Q))
    
    Args:
        p (torch.Tensor): First probability distribution (P)
        q (torch.Tensor): Second probability distribution (Q) 
        epsilon (float): Small value to avoid log(0)
    
    Returns:
        torch.Tensor: KL divergence for each position/token
    """
    # Add small epsilon to avoid log(0)
    p_safe = p + epsilon
    q_safe = q + epsilon
    
    # Calculate KL divergence: D(P||Q) = sum(P * log(P/Q))
    kl_div = p_safe * torch.log(p_safe / q_safe)
    
    # Sum over vocabulary dimension (last dimension)
    kl_div = kl_div.sum(dim=-1)
    
    return kl_div

In [21]:
import plotly.io as pio

# Check current default renderer
print(f"Current default renderer: {pio.renderers.default}")

# List available renderers
print(f"Available renderers: {list(pio.renderers)}")

# Set the renderer explicitly for Jupyter
pio.renderers.default = "notebook"  # or try "iframe" or "notebook_connected"

def quick_kl_plot(kl_div_0_1, kl_div_1_0, test_sequence, tokenizer):
    """Quick version with minimal customization."""
    # Get tokens for hover
    input_ids = tokenizer.encode(test_sequence.strip(), add_special_tokens=False, return_tensors="pt")
    tokens = [tokenizer.decode([tid]) for tid in input_ids[0]]
    
    # Convert to numpy
    kl_0_1_np = to_numpy(kl_div_0_1)
    kl_1_0_np = to_numpy(kl_div_1_0)
    
    # Create simple plot
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        y=kl_0_1_np, 
        name='D(Context||No Context)',
        text=tokens[max_length//2:max_length],
        hovertemplate='Token: %{text}<br>KL Div: %{y:.6f}<extra></extra>'
    ))
    fig.add_trace(go.Scatter(
        y=kl_1_0_np,
        name='D(No Context||Context)', 
        text=tokens[max_length//2:max_length],
        hovertemplate='Token: %{text}<br>KL Div: %{y:.6f}<extra></extra>'
    ))
    fig.update_layout(title="KL Divergence by Token", xaxis_title="Token Position", yaxis_title="KL Divergence")
    return fig

# Uncomment to use the quick version:
quick_fig = quick_kl_plot(kl_div_0_1, kl_div_1_0, test_sequence, analyzer.tokenizer)
quick_fig.show()

Current default renderer: plotly_mimetype
Available renderers: ['plotly_mimetype', 'jupyterlab', 'nteract', 'vscode', 'notebook', 'notebook_connected', 'kaggle', 'azure', 'colab', 'cocalc', 'databricks', 'json', 'png', 'jpeg', 'jpg', 'svg', 'pdf', 'browser', 'firefox', 'chrome', 'chromium', 'iframe', 'iframe_connected', 'sphinx_gallery', 'sphinx_gallery_png']


NameError: name 'kl_div_0_1' is not defined

In [17]:

import plotly.graph_objects as go

def quick_kl_plot_widget(kl_div_0_1, kl_div_1_0, test_sequence, tokenizer):
    """Version using FigureWidget for better Jupyter integration."""
    # Get tokens for hover
    input_ids = tokenizer.encode(test_sequence.strip(), add_special_tokens=False, return_tensors="pt")
    tokens = [tokenizer.decode([tid]) for tid in input_ids[0]]
    
    # Convert to numpy
    kl_0_1_np = to_numpy(kl_div_0_1)
    kl_1_0_np = to_numpy(kl_div_1_0)
    
    # Create FigureWidget instead of Figure
    fig = go.FigureWidget()
    
    fig.add_trace(go.Scatter(
        y=kl_0_1_np, 
        name='D(Context||No Context)',
        text=tokens,
        hovertemplate='Token: %{text}<br>KL Div: %{y:.6f}<extra></extra>'
    ))
    fig.add_trace(go.Scatter(
        y=kl_1_0_np,
        name='D(No Context||Context)', 
        text=tokens,
        hovertemplate='Token: %{text}<br>KL Div: %{y:.6f}<extra></extra>'
    ))
    fig.update_layout(
        title="KL Divergence by Token", 
        xaxis_title="Token Position", 
        yaxis_title="KL Divergence",
        height=500,
        width=900
    )
    
    return fig

# Create and display the widget
fig_widget = quick_kl_plot_widget(kl_div_0_1, kl_div_1_0, test_sequence, analyzer.tokenizer)

# Just return it - Jupyter should display it automatically
fig_widget

FigureWidget({
    'data': [{'hovertemplate': 'Token: %{text}<br>KL Div: %{y:.6f}<extra></extra>',
              'name': 'D(Context||No Context)',
              'text': [Tra, iler,  , ...,  great,  performance, .],
              'type': 'scatter',
              'uid': '3d2d000f-4052-4c58-9901-1b7261dd1fe2',
              'y': array([1.73446989e+00, 2.42969084e+00, 2.73149157e+00, 3.83102894e-01,
                          3.88169646e-01, 6.49218913e-03, 9.28283453e-01, 1.22559500e+00,
                          1.27089751e+00, 5.73200107e-01, 1.76804766e-01, 8.16521585e-01,
                          3.52075338e-01, 4.77327585e-01, 7.80172125e-02, 1.86009258e-01,
                          1.72385067e-01, 1.83130860e-01, 2.08399937e-01, 1.56354755e-01,
                          3.53573084e-01, 3.06450836e-02, 1.88902572e-01, 4.50336814e-01,
                          1.06460318e-01, 5.39604761e-02, 6.90980703e-02, 9.52870399e-02,
                          6.02687597e-02, 1.54729366e-01, 2.2

In [19]:
import plotly.graph_objects as go

# Super simple test
test = go.Figure(data=[go.Bar(x=[1, 2, 3], y=[4, 5, 6])])
test.show()

In [22]:

try:
    import os
    # Check for JupyterLab
    if 'JUPYTERLAB_DIR' in os.environ:
        print("You're likely using JupyterLab")
    else:
        print("You're likely using Jupyter Notebook (classic)")
except:
    pass

# Method 2: Check IPython class
try:
    ipython = get_ipython()
    print(f"IPython class: {ipython.__class__.__module__}.{ipython.__class__.__name__}")
    
    if 'ZMQInteractiveShell' in str(ipython.__class__):
        print("Running in Jupyter (Notebook or Lab)")
        
        # Check for specific indicators
        if hasattr(ipython, 'kernel'):
            print(f"Kernel: {ipython.kernel}")
            
except Exception as e:
    print(f"Could not determine environment: {e}")

# Method 3: Check Jupyter version info
import subprocess
try:
    result = subprocess.run(['jupyter', '--version'], capture_output=True, text=True)
    print("\nJupyter version info:")
    print(result.stdout)
except:
    print("Could not run jupyter --version")

You're likely using JupyterLab
IPython class: ipykernel.zmqshell.ZMQInteractiveShell
Running in Jupyter (Notebook or Lab)
Kernel: <ipykernel.ipkernel.IPythonKernel object at 0x14952b28eed0>

Jupyter version info:
Selected Jupyter core packages...
IPython          : 8.5.0
ipykernel        : 6.13.0
ipywidgets       : 7.7.4
jupyter_client   : 7.3.1
jupyter_core     : 4.10.0
jupyter_server   : 1.21.0
jupyterlab       : 3.5.0
nbclient         : 0.6.3
nbconvert        : 6.5.3
nbformat         : 5.4.0
notebook         : 6.4.0
qtconsole        : not installed
traitlets        : 5.2.1.post0



In [12]:
def quick_kl_plot(kl_div_0_1, kl_div_1_0, test_sequence, tokenizer):
    """Quick version with minimal customization."""
    # Get tokens for hover
    input_ids = tokenizer.encode(test_sequence.strip(), add_special_tokens=False, return_tensors="pt")
    tokens = [tokenizer.decode([tid]) for tid in input_ids[0]]
    
    # Convert to numpy
    kl_0_1_np = to_numpy(kl_div_0_1)
    kl_1_0_np = to_numpy(kl_div_1_0)
    
    # Create simple plot
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        y=kl_0_1_np, 
        name='D(Context||No Context)',
        text=tokens,
        hovertemplate='Token: %{text}<br>KL Div: %{y:.6f}<extra></extra>'
    ))
    fig.add_trace(go.Scatter(
        y=kl_1_0_np,
        name='D(No Context||Context)', 
        text=tokens,
        hovertemplate='Token: %{text}<br>KL Div: %{y:.6f}<extra></extra>'
    ))
    fig.update_layout(title="KL Divergence by Token", xaxis_title="Token Position", yaxis_title="KL Divergence")
    return fig

# Create the figure
quick_fig = quick_kl_plot(kl_div_0_1, kl_div_1_0, test_sequence, analyzer.tokenizer)

# Alternative display methods:

# Option 1: Save to HTML file and open in browser
# quick_fig.write_html("kl_divergence_plot.html")
# print("Plot saved to kl_divergence_plot.html")


display(quick_fig)



